# Santander: data exploration

We want to predict which products a customer will acquire next month. Each row records one customer's profile and product holdings for a month.

Start with the full training file, then select 5,000 customer IDs with seed 42 and keep all their months. The competition test file has no product labels and is not used here.

[Data source](https://www.kaggle.com/competitions/santander-product-recommendation/data) · [Download instructions](../README.md) · [Buyer classification](classification_eda.ipynb) · [Model comparison](modeling.ipynb)

Use a fresh kernel and run one notebook at a time. This notebook now uses 5,000 customers to reduce memory use. The saved 50,000-customer benchmark results are separate.

In [ ]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import polars as pl
import matplotlib.pyplot as plt
from recommender_santander.utils import get_project_root
from recommender_santander.columns import COLUMN_NAMES, PRODUCTS
from recommender_santander.data import training_csv, load_sample, monthly_pairs

PROJECT_DIR = get_project_root()
source = PROJECT_DIR / "data/raw/train/train_ver2.csv.zip"
if not source.exists():
    source = source.with_suffix("")
pl.Config.set_tbl_rows(12)

## Size and monthly coverage

Use a lazy scan for the full CSV to avoid loading it all into memory. Read mixed-type fields as text initially, strip spaces, and normalize missing markers.

In [ ]:
raw = pl.scan_csv(training_csv(source), infer_schema=False).rename(COLUMN_NAMES)
raw = raw.with_columns(pl.all().str.strip_chars())
raw = raw.with_columns([
    pl.when(pl.col(column).is_in(["", "NA", "N/A", "null", "None", "nan"]))
    .then(None).otherwise(pl.col(column)).alias(column)
    for column in COLUMN_NAMES.values()
])
monthly = raw.group_by("snapshot_date").agg(
    pl.len().alias("rows"), pl.col("customer_id").n_unique().alias("customers"),
).sort("snapshot_date").collect(engine="streaming")
customer_count = raw.select(pl.col("customer_id").n_unique()).collect(engine="streaming").item()
print(f"Rows: {monthly['rows'].sum():,}; customers: {customer_count:,}; months: {monthly.height}")

In [ ]:
monthly.to_pandas().plot(x="snapshot_date", y="customers", marker="o", legend=False, rot=45, figsize=(9, 3))
plt.ylabel("Observed customers")
plt.xlabel("Month")
plt.tight_layout()
plt.show()

There are 17 months of training data. The jump in July 2015 may be a change in coverage; these counts do not tell us its cause. We will use a time-based split for modeling.

## Missing values and product indicators

In [ ]:
counts = raw.select(pl.all().null_count()).collect(engine="streaming")
missing = counts.unpivot(variable_name="column", value_name="missing_rows")
missing = missing.with_columns(
    (pl.col("missing_rows") / monthly["rows"].sum() * 100).round(2).alias("missing_pct")
).sort("missing_pct", descending=True)
display(missing.head(10).to_pandas())

In [ ]:
missing.head(8).sort("missing_pct").to_pandas().plot.barh(
    x="column", y="missing_pct", legend=False, figsize=(9, 3),
)
plt.xlabel("Missing records (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
invalid_products = raw.select([
    (~pl.col(product).is_in(["0", "1", "0.0", "1.0"]))
    .fill_null(False).sum().alias(product)
    for product in PRODUCTS
]).collect(engine="streaming")
print("Unexpected product values:", sum(invalid_products.row(0)))
print("Duplicate customer-months:", (monthly["rows"] - monthly["customers"]).sum())

Employee spouse status and last primary-customer date are almost empty, so they are not model inputs. Income is missing in about 20% of rows. Missing product holdings mean unknown ownership, not zero.

The shared loader below converts dates and numbers. It treats negative income/tenure and ages outside 0–120 as missing. Those age limits are a cleaning choice.

## A fixed customer sample

Sample customer IDs rather than individual rows, so their histories stay together. The sample is built from the raw file on each run; it is not saved to a processed-data cache.

In [ ]:
sample = load_sample(source, customers=5_000, seed=42)
print(f"Sample: {sample.height:,} rows, {sample['customer_id'].n_unique():,} customers")
display(sample.select("age", "tenure_months", "household_income").describe().to_pandas())

In [ ]:
history_lengths = sample.group_by("customer_id").len(name="months_observed")
history_counts = history_lengths.group_by("months_observed").len(name="customers").sort("months_observed")
history_counts.to_pandas().plot.bar(x="months_observed", y="customers", legend=False, figsize=(9, 3))
plt.xlabel("Observed months per customer")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

Not every customer has 17 observations. An absent month cannot be treated as zero product ownership.

## Ownership versus new acquisitions

A new acquisition is a change from 0 to 1 between consecutive months. `monthly_pairs` uses the earlier profile as features and drops gaps or pairs with unknown product holdings.

In [ ]:
pairs = monthly_pairs(sample)
print(f"Complete consecutive pairs: {pairs.height:,}")
ownership = sample.select([pl.col(product).mean().alias(product) for product in PRODUCTS])
ownership = ownership.unpivot(variable_name="product", value_name="ownership_rate")
acquisitions = pairs.select([
    pl.col(f"new__{product}").sum().alias(product) for product in PRODUCTS
]).unpivot(variable_name="product", value_name="acquisitions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
held = ownership.sort("ownership_rate").tail(8).to_pandas().set_index("product") * 100
added = acquisitions.sort("acquisitions").tail(8).to_pandas().set_index("product")
held.plot.barh(ax=axes[0], legend=False, title="Most held products")
added.plot.barh(ax=axes[1], legend=False, title="Most acquired products")
axes[0].set_xlabel("Customer-month ownership (%)")
axes[1].set_xlabel("New acquisitions")
plt.tight_layout()
plt.show()

In [ ]:
target_counts = pairs.with_columns(
    pl.sum_horizontal([f"new__{product}" for product in PRODUCTS]).alias("new_products")
).group_by("new_products").len(name="pairs").sort("new_products")
display(target_counts.to_pandas())
buyers = target_counts.filter(pl.col("new_products") > 0)["pairs"].sum()
print(f"Pairs with any new acquisition: {100 * buyers / pairs.height:.2f}%")

Current accounts are widely held, while direct debit and payroll-related products account for many acquisitions. Most pairs have no new acquisitions, so accuracy alone would be misleading.

[Classification EDA](classification_eda.ipynb) looks at who acquires something. [Modeling](modeling.ipynb) compares product rankings and explains MAP@7.